# 09. Benchmark семантического поиска и дедупликации Top-10

Smoke-test проверяет два требования из презентации:

- `MRR@10 > keyword baseline` для семантического поиска;
- снижение доли повторов в Top-10 не менее чем на 30% относительно baseline.

Запросом служит заголовок golden-публикации. Релевантными считаются другие публикации того же экспертного `cluster_id`. Сам документ-запрос исключается. Для дедупликации используются **предсказанные** кластеры финального `exp_10a`, а повторы оцениваются по независимым golden-кластерам.

Это технический smoke-test на синтетических запросах, а не продуктовая оценка на реальных запросах клиента.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / '.git').exists() and (path / 'README.md').exists():
            return path
    raise RuntimeError('Project root not found')


PROJECT_DIR = find_project_root()
SRC_DIR = PROJECT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

GOLDEN_PATH = PROJECT_DIR / 'data/prepared/lenta_golden_set_human.csv'
POOL_PATH = PROJECT_DIR / 'data/prepared/lenta_golden_candidate_pool.csv'
PREDICTION_PATH = PROJECT_DIR / 'data/predictions/exp_10a_current_model_on_exp10_clustering.csv'
DOCUMENT_EMBEDDINGS_PATH = PROJECT_DIR / 'data/artifacts/embeddings/bge_m3_candidate_pool_id_aligned.npz'
QUERY_EMBEDDINGS_PATH = PROJECT_DIR / 'data/artifacts/embeddings/search_benchmark_title_queries_bge_m3.npz'
OUTPUT_DIR = PROJECT_DIR / 'data/artifacts/search_benchmark'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_K = 10
MODEL_NAME = 'BAAI/bge-m3'

print('PROJECT_DIR:', PROJECT_DIR)
print('GOLDEN_PATH:', GOLDEN_PATH)
print('PREDICTION_PATH:', PREDICTION_PATH)

PROJECT_DIR: E:\ML\Projects\Git\news-flow-analysis
GOLDEN_PATH: E:\ML\Projects\Git\news-flow-analysis\data\prepared\lenta_golden_set_human.csv
PREDICTION_PATH: E:\ML\Projects\Git\news-flow-analysis\data\predictions\exp_10a_current_model_on_exp10_clustering.csv


## Данные

Корпус ограничен golden-набором, потому что только для него есть полная экспертная разметка сюжетов. Иначе неразмеченные релевантные документы стали бы ложными отрицательными примерами.

In [2]:
golden = pd.read_csv(GOLDEN_PATH)
pool = pd.read_csv(POOL_PATH)
prediction = pd.read_csv(PREDICTION_PATH, usecols=['news_id', 'cluster_id'])

for frame in [golden, pool, prediction]:
    frame['news_id'] = frame['news_id'].astype(str)

corpus = (
    golden[['news_id', 'published_at', 'topic', 'title', 'text', 'cluster_id']]
    .rename(columns={'cluster_id': 'gold_cluster_id'})
    .merge(pool[['news_id', 'model_text']], on='news_id', how='left', validate='one_to_one')
    .merge(
        prediction.rename(columns={'cluster_id': 'predicted_cluster_id'}),
        on='news_id',
        how='left',
        validate='one_to_one',
    )
    .sort_values(['published_at', 'news_id'])
    .reset_index(drop=True)
)

if corpus[['model_text', 'predicted_cluster_id']].isna().any().any():
    raise ValueError('Missing model_text or predicted_cluster_id after merge')

cluster_sizes = corpus['gold_cluster_id'].value_counts()
eligible_mask = corpus['gold_cluster_id'].map(cluster_sizes).gt(1)
query_positions = np.flatnonzero(eligible_mask.to_numpy())

print('Corpus rows:', len(corpus))
print('Golden clusters:', corpus['gold_cluster_id'].nunique())
print('Multi-document clusters:', int((cluster_sizes > 1).sum()))
print('Eligible title queries:', len(query_positions))
corpus.head(3)

Corpus rows: 121
Golden clusters: 34
Multi-document clusters: 32
Eligible title queries: 119


,news_id,published_at,topic,title,text,gold_cluster_id,model_text,predicted_cluster_id
0,88546,2004-03-01,Экономика,Саудовская Аравия пообещала не допустить нехва...,Саудовская Аравия не допустит нехватки нефти н...,prelim_0051,Саудовская Аравия пообещала не допустить нехва...,baseline_cluster_00011
1,88572,2004-03-01,Россия,Больше всех на предвыборную кампанию потратил ...,"По информации Центризбиркома России, больше вс...",prelim_0036,Больше всех на предвыборную кампанию потратил ...,baseline_cluster_00022
2,88579,2004-03-01,Мир,На Гаити высадились французские миротворцы,На Гаити высадился передовой отряд французских...,prelim_0013,На Гаити высадились французские миротворцы. На...,baseline_cluster_00028


## Ранжирование

- keyword baseline: word-level TF-IDF по `model_text`;
- semantic: cosine similarity между BGE-M3 embedding заголовка и документа;
- semantic + dedup: тот же semantic ranking, но остаётся первый документ каждого предсказанного `exp_10a` кластера.

In [3]:
from model.embeddings import load_id_aligned_embeddings

document_embeddings = load_id_aligned_embeddings(
    DOCUMENT_EMBEDDINGS_PATH,
    corpus['news_id'],
)
document_embeddings = normalize(document_embeddings, norm='l2', axis=1)

query_ids = corpus.iloc[query_positions]['news_id'].astype(str).to_numpy()
query_titles = corpus.iloc[query_positions]['title'].fillna('').astype(str).tolist()

if QUERY_EMBEDDINGS_PATH.exists():
    with np.load(QUERY_EMBEDDINGS_PATH, allow_pickle=True) as cache:
        cached_ids = cache['ids'].astype(str)
        if np.array_equal(cached_ids, query_ids):
            query_embeddings = cache['embeddings'].astype(np.float32)
        else:
            query_embeddings = None
else:
    query_embeddings = None

if query_embeddings is None:
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer(MODEL_NAME)
    query_embeddings = model.encode(
        query_titles,
        batch_size=16,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    ).astype(np.float32)
    np.savez_compressed(QUERY_EMBEDDINGS_PATH, ids=query_ids, embeddings=query_embeddings)

query_embeddings = normalize(query_embeddings, norm='l2', axis=1)
semantic_scores = np.asarray(query_embeddings @ document_embeddings.T)

tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
)
document_tfidf = tfidf.fit_transform(corpus['model_text'].fillna('').astype(str))
query_tfidf = tfidf.transform(query_titles)
keyword_scores = (query_tfidf @ document_tfidf.T).toarray()

print('Document embeddings:', document_embeddings.shape)
print('Query embeddings:', query_embeddings.shape)
print('TF-IDF vocabulary:', len(tfidf.vocabulary_))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Document embeddings: (121, 1024)
Query embeddings: (119, 1024)
TF-IDF vocabulary: 23369


In [4]:
def rank_indices(scores: np.ndarray, self_position: int) -> list[int]:
    order = np.argsort(-scores, kind='stable').tolist()
    return [idx for idx in order if idx != self_position]


def collapse_by_predicted_cluster(ranking: list[int], top_k: int) -> list[int]:
    selected = []
    seen = set()
    predicted = corpus['predicted_cluster_id'].astype(str).to_numpy()
    for idx in ranking:
        cluster_id = predicted[idx]
        if cluster_id in seen:
            continue
        seen.add(cluster_id)
        selected.append(idx)
        if len(selected) == top_k:
            break
    return selected


def evaluate_ranking(ranking: list[int], query_position: int, top_k: int) -> dict:
    top = ranking[:top_k]
    query_cluster = str(corpus.iloc[query_position]['gold_cluster_id'])
    result_clusters = corpus.iloc[top]['gold_cluster_id'].astype(str).tolist()
    relevant = [cluster_id == query_cluster for cluster_id in result_clusters]
    relevant_ranks = [rank for rank, flag in enumerate(relevant, start=1) if flag]
    reciprocal_rank = 1.0 / relevant_ranks[0] if relevant_ranks else 0.0

    seen = set()
    repeated_results = 0
    for cluster_id in result_clusters:
        if cluster_id in seen:
            repeated_results += 1
        else:
            seen.add(cluster_id)

    return {
        'reciprocal_rank_at_10': reciprocal_rank,
        'hit_at_10': float(bool(relevant_ranks)),
        'relevant_at_10': int(sum(relevant)),
        'duplicate_count_at_10': repeated_results,
        'duplicate_rate_at_10': repeated_results / len(top) if top else 0.0,
        'unique_gold_clusters_at_10': len(set(result_clusters)),
    }


rows = []
rankings = {}
for query_row, query_position in enumerate(query_positions):
    keyword_ranking = rank_indices(keyword_scores[query_row], query_position)
    semantic_ranking = rank_indices(semantic_scores[query_row], query_position)
    collapsed_ranking = collapse_by_predicted_cluster(semantic_ranking, TOP_K)

    method_rankings = {
        'keyword_tfidf': keyword_ranking,
        'bge_m3': semantic_ranking,
        'bge_m3_exp10_dedup': collapsed_ranking,
    }
    rankings[str(corpus.iloc[query_position]['news_id'])] = method_rankings

    for method, ranking in method_rankings.items():
        rows.append(
            {
                'query_news_id': corpus.iloc[query_position]['news_id'],
                'query_title': corpus.iloc[query_position]['title'],
                'query_gold_cluster_id': corpus.iloc[query_position]['gold_cluster_id'],
                'method': method,
                **evaluate_ranking(ranking, query_position, TOP_K),
            }
        )

per_query = pd.DataFrame(rows)
summary = (
    per_query.groupby('method', as_index=False)
    .agg(
        queries=('query_news_id', 'count'),
        mrr_at_10=('reciprocal_rank_at_10', 'mean'),
        hit_rate_at_10=('hit_at_10', 'mean'),
        mean_relevant_at_10=('relevant_at_10', 'mean'),
        duplicate_rate_at_10=('duplicate_rate_at_10', 'mean'),
        mean_duplicate_count_at_10=('duplicate_count_at_10', 'mean'),
        mean_unique_gold_clusters_at_10=('unique_gold_clusters_at_10', 'mean'),
    )
)

keyword_duplicate_rate = float(
    summary.loc[summary['method'].eq('keyword_tfidf'), 'duplicate_rate_at_10'].iloc[0]
)
summary['duplicate_reduction_vs_keyword'] = (
    1.0 - summary['duplicate_rate_at_10'] / keyword_duplicate_rate
)
summary

,method,queries,mrr_at_10,hit_rate_at_10,mean_relevant_at_10,duplicate_rate_at_10,mean_duplicate_count_at_10,mean_unique_gold_clusters_at_10,duplicate_reduction_vs_keyword
0,bge_m3,119,0.992647,1.000000,2.924370,0.515126,5.151261,4.848739,-0.104505
1,bge_m3_exp10_dedup,119,0.994398,1.000000,1.277311,0.115966,1.159664,8.840336,0.751351
2,keyword_tfidf,119,0.906663,0.991597,2.806723,0.466387,4.663866,5.336134,0.000000


## Проверка требований и сохранение результатов

In [5]:
keyword_row = summary.set_index('method').loc['keyword_tfidf']
semantic_row = summary.set_index('method').loc['bge_m3']
dedup_row = summary.set_index('method').loc['bge_m3_exp10_dedup']

checks = {
    'mrr_at_10_semantic_above_keyword': bool(
        semantic_row['mrr_at_10'] > keyword_row['mrr_at_10']
    ),
    'dedup_reduction_at_least_30_percent': bool(
        dedup_row['duplicate_reduction_vs_keyword'] >= 0.30
    ),
}

summary_path = OUTPUT_DIR / 'search_benchmark_summary.csv'
per_query_path = OUTPUT_DIR / 'search_benchmark_per_query.csv'
checks_path = OUTPUT_DIR / 'search_benchmark_checks.json'
summary.to_csv(summary_path, index=False)
per_query.to_csv(per_query_path, index=False)
checks_path.write_text(json.dumps(checks, ensure_ascii=False, indent=2), encoding='utf-8')

print(summary.to_string(index=False))
print('\nChecks:', json.dumps(checks, ensure_ascii=False, indent=2))
print('\nSaved:', summary_path)
print('Saved:', per_query_path)
print('Saved:', checks_path)

            method  queries  mrr_at_10  hit_rate_at_10  mean_relevant_at_10  duplicate_rate_at_10  mean_duplicate_count_at_10  mean_unique_gold_clusters_at_10  duplicate_reduction_vs_keyword
            bge_m3      119   0.992647        1.000000             2.924370              0.515126                    5.151261                         4.848739                       -0.104505
bge_m3_exp10_dedup      119   0.994398        1.000000             1.277311              0.115966                    1.159664                         8.840336                        0.751351
     keyword_tfidf      119   0.906663        0.991597             2.806723              0.466387                    4.663866                         5.336134                        0.000000

Checks: {
  "mrr_at_10_semantic_above_keyword": true,
  "dedup_reduction_at_least_30_percent": true
}

Saved: E:\ML\Projects\Git\news-flow-analysis\data\artifacts\search_benchmark\search_benchmark_summary.csv
Saved: E:\ML\Projects\Git\

## Интерпретация

- Если BGE-M3 превосходит TF-IDF по `MRR@10`, выполнен технический search benchmark из презентации.
- `duplicate_rate_at_10` — доля результатов, которые повторяют уже представленный в Top-10 golden-сюжет.
- `duplicate_reduction_vs_keyword` сравнивает эту долю с TF-IDF baseline.
- Collapse опирается на предсказанные `exp_10a` кластеры, поэтому ошибки кластеризации влияют на результат честно.

Ограничения: запросы получены из заголовков документов, корпус мал и состоит только из golden-набора. Для продуктовой проверки нужны реальные запросы аналитиков и отдельная экспертная разметка релевантности и дублей.